# 4 - GOLD: 
## Aplicação de Regras de Negócio

>> RAW (CSV original)
        │
        ▼
>> BRONZE (Parquet 1:1 com o CSV)
        │
        ▼
>> SILVER (dados padronizados, limpos e enriquecidos)
        │
        ▼
>  GOLD (indicadores e agregações)

## 4.0 - Bibliotecas

In [15]:
from dotenv import load_dotenv
from datetime import datetime
from src.data.utils import *

load_dotenv()
session = iniciar_cessao_aws()
s3 = session.client("s3")
BUCKET = os.getenv("BUCKET_NAME")


## 4.1 Concatenando as tabelas, independente do ano

In [16]:
TABELAS = [
    "TS_ALUNO",
    "TS_ESTADO",
    "TS_MUNICIPIO",
    "TS_ITEM",
    "RESULTADOS_METAS",
    "RESULTADOS_METAS_MUNICIPIO"
]

# anos de interesse - poderia pegar diretamente do nome do arquivo, mas para fins de teste, vamos filtrar por ano
ANOS = [2023, 2024, 2025]

In [19]:
for tabela in TABELAS:

    print(f"\n{'='*60}")
    print(f"Consolidando {tabela}")
    print(f"{'='*60}")

    dfs = []

    for ano in ANOS:

        print(f"Lendo {tabela} - {ano}")

        df = carregar_parquet_s3(
            s3_client=s3,
            bucket=BUCKET,
            ano=ano,
            nome_tabela=tabela,
            camada="silver"
        )

        # Adiciona metadado do ano
        df["ANO"] = ano

        dfs.append(df)

    # Consolida os anos
    df_gold = pd.concat(
        dfs,
        ignore_index=True,
        sort=False
    )

    print(f"Total de registros: {len(df_gold):,}")

    # Salva na Gold
    salvar_parquet_s3(
        s3_client=s3,
        bucket=BUCKET,
        df=df_gold,
        ano="historico",
        tabela=tabela,
        camada="gold"
    )

    print(f"{tabela} consolidada com sucesso.")


Consolidando TS_ALUNO
Lendo TS_ALUNO - 2023
Lendo: s3://fiap-postech-challenge-datascience-002/silver/ano=2023/dados/TS_ALUNO.parquet
Lendo TS_ALUNO - 2024
Lendo: s3://fiap-postech-challenge-datascience-002/silver/ano=2024/dados/TS_ALUNO.parquet
Lendo TS_ALUNO - 2025
Lendo: s3://fiap-postech-challenge-datascience-002/silver/ano=2025/dados/TS_ALUNO.parquet
Total de registros: 6,090,791
Arquivo salvo: s3://fiap-postech-challenge-datascience-002/gold/ano=historico/dados/TS_ALUNO.parquet
TS_ALUNO consolidada com sucesso.

Consolidando TS_ESTADO
Lendo TS_ESTADO - 2023
Lendo: s3://fiap-postech-challenge-datascience-002/silver/ano=2023/dados/TS_ESTADO.parquet
Lendo TS_ESTADO - 2024
Lendo: s3://fiap-postech-challenge-datascience-002/silver/ano=2024/dados/TS_ESTADO.parquet
Lendo TS_ESTADO - 2025
Lendo: s3://fiap-postech-challenge-datascience-002/silver/ano=2025/dados/TS_ESTADO.parquet
Total de registros: 225
Arquivo salvo: s3://fiap-postech-challenge-datascience-002/gold/ano=historico/dados/TS

### 4.1.1 - Verificando a carga na consolidada na Gold

In [20]:
pd.set_option('display.max_columns', None)
df_alunos = carregar_parquet_s3(
    s3_client=s3,
    bucket=BUCKET,
    ano="historico",
    nome_tabela="TS_ALUNO",
    camada="gold"
)



df_municipios = carregar_parquet_s3(
    s3_client=s3,
    bucket=BUCKET,
    ano="historico",
    nome_tabela="TS_MUNICIPIO",
    camada="gold"
)



df_estados = carregar_parquet_s3(
    s3_client=s3,
    bucket=BUCKET,
    ano="historico",
    nome_tabela="TS_ESTADO",
    camada="gold"
)



df_itens = carregar_parquet_s3(
    s3_client=s3,
    bucket=BUCKET,
    ano="historico",
    nome_tabela="TS_ITEM",
    camada="gold"
)



df_resultados_metas = carregar_parquet_s3(
    s3_client=s3,
    bucket=BUCKET,
    ano="historico",
    nome_tabela="RESULTADOS_METAS",
    camada="gold"
)



df_resultados_metas_municipios = carregar_parquet_s3(
    s3_client=s3,
    bucket=BUCKET,
    ano="historico",
    nome_tabela="RESULTADOS_METAS_MUNICIPIO",
    camada="gold"
)



Lendo: s3://fiap-postech-challenge-datascience-002/gold/ano=historico/dados/TS_ALUNO.parquet
Lendo: s3://fiap-postech-challenge-datascience-002/gold/ano=historico/dados/TS_MUNICIPIO.parquet
Lendo: s3://fiap-postech-challenge-datascience-002/gold/ano=historico/dados/TS_ESTADO.parquet
Lendo: s3://fiap-postech-challenge-datascience-002/gold/ano=historico/dados/TS_ITEM.parquet
Lendo: s3://fiap-postech-challenge-datascience-002/gold/ano=historico/dados/RESULTADOS_METAS.parquet
Lendo: s3://fiap-postech-challenge-datascience-002/gold/ano=historico/dados/RESULTADOS_METAS_MUNICIPIO.parquet


#### Verificando as tabelas

In [7]:
df_alunos.head(50) 

,NU_ANO_AVALIACAO,CO_UF,SG_UF,ID_ALUNO,TP_SERIE,ID_ESCOLA,TP_DEPENDENCIA,CO_MUNICIPIO,NO_MUNICIPIO,IN_PRESENCA_LP,IN_PREENCHIMENTO_LP,CO_CADERNO_LP,VL_PESO_ALUNO_LP,VL_PROFICIENCIA_LP,IN_ALFABETIZADO,DESC_DEPENDENCIA,DESC_SERIE,ANO,CO_BLOCO_1,TX_RESPOSTA_BLOCO_1,TX_GABARITO_BLOCO_1,CO_BLOCO_2,TX_RESPOSTA_BLOCO_2,TX_GABARITO_BLOCO_2,CO_BLOCO_3,TX_RESPOSTA_BLOCO_3,TX_GABARITO_BLOCO_3,CO_BLOCO_4,TX_RESPOSTA_BLOCO_4,TX_GABARITO_BLOCO_4
0,2023,11,RO,11008701,2,60000001,3,1100205,Porto Velho,0,0,17,NaN,NaN,0,MUNICIPAL,2º Ano Do Ensino Fundamental,2023,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,2023,11,RO,11008695,2,60000001,3,1100205,Porto Velho,1,1,17,1.045465,714.314857,0,MUNICIPAL,2º Ano Do Ensino Fundamental,2023,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,2023,11,RO,11008687,2,60000001,3,1100205,Porto Velho,0,0,17,NaN,NaN,0,MUNICIPAL,2º Ano Do Ensino Fundamental,2023,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,2023,11,RO,11008682,2,60000001,3,1100205,Porto Velho,1,1,17,1.045465,759.206313,1,MUNICIPAL,2º Ano Do Ensino Fundamental,2023,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,2023,11,RO,11008729,2,60000001,3,1100205,Porto Velho,0,0,19,NaN,NaN,0,MUNICIPAL,2º Ano Do Ensino Fundamental,2023,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [7]:
df_municipios.head() 

,NU_ANO_AVALIACAO,CO_UF,SG_UF,CO_MUNICIPIO,NO_MUNICIPIO,TP_SERIE,ID_TIPO_REDE,PC_ALUNO_ALFABETIZADO,VL_MEDIA_LP,DESC_SERIE,ANO,PC_ALUNO_NIVEL_0_LP,PC_ALUNO_NIVEL_1_LP,PC_ALUNO_NIVEL_2_LP,PC_ALUNO_NIVEL_3_LP,PC_ALUNO_NIVEL_4_LP,PC_ALUNO_NIVEL_5_LP,PC_ALUNO_NIVEL_6_LP,PC_ALUNO_NIVEL_7_LP,PC_ALUNO_NIVEL_8_LP
0,2023,11,RO,1100015,Alta Floresta D'Oeste,2,5,64.55,758.3304,2º Ano Do Ensino Fundamental,2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023,11,RO,1100015,Alta Floresta D'Oeste,2,3,64.55,758.3304,2º Ano Do Ensino Fundamental,2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023,11,RO,1100023,Ariquemes,2,3,62.30,757.0999,2º Ano Do Ensino Fundamental,2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2023,11,RO,1100023,Ariquemes,2,5,62.30,757.0999,2º Ano Do Ensino Fundamental,2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2023,11,RO,1100031,Cabixi,2,5,69.10,767.8763,2º Ano Do Ensino Fundamental,2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
df_estados.head() 

,NU_ANO_AVALIACAO,CO_UF,SG_UF,TP_SERIE,ID_TIPO_REDE,PC_ALUNO_ALFABETIZADO,VL_MEDIA_LP,DESC_SERIE,ANO,PC_ALUNO_NIVEL_0_LP,PC_ALUNO_NIVEL_1_LP,PC_ALUNO_NIVEL_2_LP,PC_ALUNO_NIVEL_3_LP,PC_ALUNO_NIVEL_4_LP,PC_ALUNO_NIVEL_5_LP,PC_ALUNO_NIVEL_6_LP,PC_ALUNO_NIVEL_7_LP,PC_ALUNO_NIVEL_8_LP
0,2023,11,RO,2,2,58.65,751.4731,2º Ano Do Ensino Fundamental,2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023,11,RO,2,3,65.17,760.1971,2º Ano Do Ensino Fundamental,2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023,11,RO,2,5,64.60,759.4357,2º Ano Do Ensino Fundamental,2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2023,13,AM,2,3,49.20,733.6637,2º Ano Do Ensino Fundamental,2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2023,13,AM,2,5,52.20,736.4687,2º Ano Do Ensino Fundamental,2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_itens.head() 

,NU_ANO_AVALIACAO,CO_UF,SG_UF,CO_BLOCO,NU_POSICAO,CO_ITEM,TP_SERIE,TP_DISCIPLINA,NU_DESCRITOR_HABILIDADE,DS_GABARITO,TP_RESPOSTA_ITEM,TP_MODELO_TRI,NU_PARAM_A,NU_PARAM_B,NU_PARAM_C,NU_PARAM_B1,NU_PARAM_B2,NU_PARAM_B3,NU_PARAM_B4,IN_ITEM_COMUM,DESC_DISCIPLINA,DESC_SERIE,ANO
0,2023,11,RO,3,2,81,2,1,H4,B,3,2,0.06,708.75,0.38,NaN,NaN,NaN,NaN,0,Língua Portuguesa,2º Ano Do Ensino Fundamental,2023
1,2023,11,RO,7,6,69,2,1,H1,A,3,2,0.02,737.60,0.25,NaN,NaN,NaN,NaN,0,Língua Portuguesa,2º Ano Do Ensino Fundamental,2023
2,2023,11,RO,5,8,112,2,1,H9,D,3,2,0.03,728.75,0.15,NaN,NaN,NaN,NaN,0,Língua Portuguesa,2º Ano Do Ensino Fundamental,2023
3,2023,11,RO,7,1,98,2,1,H2,A,3,2,0.04,643.49,0.18,NaN,NaN,NaN,NaN,0,Língua Portuguesa,2º Ano Do Ensino Fundamental,2023
4,2023,11,RO,1,5,109,2,1,H9,D,3,2,0.04,757.80,0.18,NaN,NaN,NaN,NaN,0,Língua Portuguesa,2º Ano Do Ensino Fundamental,2023


In [10]:

df_resultados_metas.head() 

,ANO,CD_UF,SIGLA_UF,NOME_UF,REDE,SAEB_2019,SAEB_2021,PC_ALUNO_ALFABETIZADO,META_FINAL_2024,META_FINAL_2025,META_FINAL_2026,META_FINAL_2027,META_FINAL_2028,META_FINAL_2029,META_FINAL_2030,PC_AVALIADOS_LP,PC_ALUNO_ALFABETIZADO_2023,PC_ALUNO_ALFABETIZADO_2024,PC_ALUNO_ALFABETIZADO_2025
0,2023,NaN,<NA>,Brasil,PÚBLICA,54.70,35.77,55.89811,59.897807,63.769871,67.471106,70.966377,74.229525,77.243526,> 80,86.00,NaN,NaN,NaN
1,2023,12.0,AC,Acre,PÚBLICA,52.87,20.05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,NaN,NaN,NaN,NaN
2,2023,27.0,AL,Alagoas,PÚBLICA,39.01,30.04,43.88000,49.700000,55.500000,61.100000,66.500000,71.500000,76.000000,> 80,92.36,NaN,NaN,NaN
3,2023,13.0,AM,Amazonas,PÚBLICA,43.83,28.76,52.20000,56.800000,61.300000,65.600000,69.600000,73.400000,76.900000,> 80,76.14,NaN,NaN,NaN
4,2023,16.0,AP,Amapá,PÚBLICA,24.77,18.77,41.56000,47.600000,53.800000,59.900000,65.600000,70.900000,75.800000,> 80,89.68,NaN,NaN,NaN


In [11]:

df_resultados_metas_municipios.head() 

,ANO,CO_UF,SG_UF,CO_MUNICIPIO,NO_MUNICIPIO,NO_TP_REDE,PC_ALUNO_ALFABETIZADO,META_FINAL_2024,META_FINAL_2025,META_FINAL_2026,META_FINAL_2027,META_FINAL_2028,META_FINAL_2029,META_FINAL_2030,NIVEIS_ALFABETIZACAO_2023,PC_AVALIADOS_LP,PC_ALUNO_ALFABETIZADO_2023,PC_ALUNO_ALFABETIZADO_2024,CO_NIVEL_ALFABETIZACAO,PC_ALUNO_ALFABETIZADO_2025
0,2023,11,RO,1100015,Alta Floresta D'Oeste,MUNICIPAL,64.55,67.078601,69.512028,71.841093,74.058634,76.159493,78.140434,80.0,3.0,89.37,NaN,NaN,NaN,NaN
1,2023,11,RO,1100023,Ariquemes,MUNICIPAL,62.30,65.216878,68.023909,70.706161,73.251889,75.652567,77.902777,80.0,3.0,89.79,NaN,NaN,NaN,NaN
2,2023,11,RO,1100031,Cabixi,MUNICIPAL,69.10,70.845029,72.530686,74.154438,75.714346,77.209042,78.637700,80.0,3.0,90.48,NaN,NaN,NaN,NaN
3,2023,11,RO,1100049,Cacoal,MUNICIPAL,62.51,65.390716,68.162817,70.811990,73.326986,75.699642,77.924781,80.0,3.0,84.44,NaN,NaN,NaN,NaN
4,2023,11,RO,1100056,Cerejeiras,MUNICIPAL,58.53,62.090401,65.525172,68.805091,71.906748,74.812904,77.512445,80.0,2.0,92.12,NaN,NaN,NaN,NaN


In [35]:
df_alunos.columns

Index(['NU_ANO_AVALIACAO', 'CO_UF', 'SG_UF', 'ID_ALUNO', 'TP_SERIE',
       'ID_ESCOLA', 'TP_DEPENDENCIA', 'CO_MUNICIPIO', 'NO_MUNICIPIO',
       'IN_PRESENCA_LP', 'IN_PREENCHIMENTO_LP', 'CO_CADERNO_LP',
       'VL_PESO_ALUNO_LP', 'VL_PROFICIENCIA_LP', 'IN_ALFABETIZADO',
       'DESC_DEPENDENCIA', 'DESC_SERIE', 'ANO', 'CO_BLOCO_1',
       'TX_RESPOSTA_BLOCO_1', 'TX_GABARITO_BLOCO_1', 'CO_BLOCO_2',
       'TX_RESPOSTA_BLOCO_2', 'TX_GABARITO_BLOCO_2', 'CO_BLOCO_3',
       'TX_RESPOSTA_BLOCO_3', 'TX_GABARITO_BLOCO_3', 'CO_BLOCO_4',
       'TX_RESPOSTA_BLOCO_4', 'TX_GABARITO_BLOCO_4'],
      dtype='str')

In [36]:
df_estados.columns

Index(['NU_ANO_AVALIACAO', 'CO_UF', 'SG_UF', 'TP_SERIE', 'ID_TIPO_REDE',
       'PC_ALUNO_ALFABETIZADO', 'VL_MEDIA_LP', 'DESC_SERIE', 'ANO',
       'PC_ALUNO_NIVEL_0_LP', 'PC_ALUNO_NIVEL_1_LP', 'PC_ALUNO_NIVEL_2_LP',
       'PC_ALUNO_NIVEL_3_LP', 'PC_ALUNO_NIVEL_4_LP', 'PC_ALUNO_NIVEL_5_LP',
       'PC_ALUNO_NIVEL_6_LP', 'PC_ALUNO_NIVEL_7_LP', 'PC_ALUNO_NIVEL_8_LP'],
      dtype='str')

In [8]:
df_municipios.columns

Index(['NU_ANO_AVALIACAO', 'CO_UF', 'SG_UF', 'CO_MUNICIPIO', 'NO_MUNICIPIO',
       'TP_SERIE', 'ID_TIPO_REDE', 'PC_ALUNO_ALFABETIZADO', 'VL_MEDIA_LP',
       'DESC_SERIE', 'ANO', 'PC_ALUNO_NIVEL_0_LP', 'PC_ALUNO_NIVEL_1_LP',
       'PC_ALUNO_NIVEL_2_LP', 'PC_ALUNO_NIVEL_3_LP', 'PC_ALUNO_NIVEL_4_LP',
       'PC_ALUNO_NIVEL_5_LP', 'PC_ALUNO_NIVEL_6_LP', 'PC_ALUNO_NIVEL_7_LP',
       'PC_ALUNO_NIVEL_8_LP'],
      dtype='str')

In [17]:
df_itens.columns

Index(['NU_ANO_AVALIACAO', 'CO_UF', 'SG_UF', 'CO_BLOCO', 'NU_POSICAO',
       'CO_ITEM', 'TP_SERIE', 'TP_DISCIPLINA', 'NU_DESCRITOR_HABILIDADE',
       'DS_GABARITO', 'TP_RESPOSTA_ITEM', 'TP_MODELO_TRI', 'NU_PARAM_A',
       'NU_PARAM_B', 'NU_PARAM_C', 'NU_PARAM_B1', 'NU_PARAM_B2', 'NU_PARAM_B3',
       'NU_PARAM_B4', 'IN_ITEM_COMUM', 'DESC_DISCIPLINA', 'DESC_SERIE', 'ANO'],
      dtype='str')

## 4.2 - Dimensões

In [21]:
# Funções
# Função para criar dimensões a partir de um DataFrame, selecionando colunas específicas.
def criar_dimensao(df, colunas):
    dim = (
        df[colunas]
        .drop_duplicates()
        .sort_values(colunas)
        .reset_index(drop=True)
    )

    return dim

In [22]:
# Municipios
DIM_MUNICIPIO = criar_dimensao(
    df_municipios,
    ["CO_UF","CO_MUNICIPIO", "NO_MUNICIPIO"]   
)

# Estados
DIM_ESTADO = criar_dimensao(
    df_estados,
    ["CO_UF", "SG_UF"]
)

## 4.3 - Tabela Fato Alfabetização


## 4.3.1 - Regra de negócios

In [23]:
# Atributos válidos:

df_alunos = df_alunos[df_alunos["TP_SERIE"] == 2]
df_municipios = df_municipios[df_municipios["TP_SERIE"] == 2]
df_estados = df_estados[df_estados["TP_SERIE"] == 2]
df_itens = df_itens[df_itens["TP_SERIE"] == 2]
df_itens = df_itens[df_itens["TP_DISCIPLINA"] == 1]


In [24]:
df_alunos = df_alunos.drop(columns=['ANO', 'CO_BLOCO_1',
       'TX_RESPOSTA_BLOCO_1', 'TX_GABARITO_BLOCO_1', 'CO_BLOCO_2',
       'TX_RESPOSTA_BLOCO_2', 'TX_GABARITO_BLOCO_2', 'CO_BLOCO_3',
       'TX_RESPOSTA_BLOCO_3', 'TX_GABARITO_BLOCO_3', 'CO_BLOCO_4',
       'TX_RESPOSTA_BLOCO_4', 'TX_GABARITO_BLOCO_4'])

In [25]:
df_alunos.head(100)

,NU_ANO_AVALIACAO,CO_UF,SG_UF,ID_ALUNO,TP_SERIE,ID_ESCOLA,TP_DEPENDENCIA,CO_MUNICIPIO,NO_MUNICIPIO,IN_PRESENCA_LP,IN_PREENCHIMENTO_LP,CO_CADERNO_LP,VL_PESO_ALUNO_LP,VL_PROFICIENCIA_LP,IN_ALFABETIZADO,DESC_DEPENDENCIA,DESC_SERIE
0,2023,11,RO,11008701,2,60000001,3,1100205,Porto Velho,0,0,17,NaN,NaN,0,MUNICIPAL,2º Ano Do Ensino Fundamental
1,2023,11,RO,11008695,2,60000001,3,1100205,Porto Velho,1,1,17,1.045465,714.314857,0,MUNICIPAL,2º Ano Do Ensino Fundamental
2,2023,11,RO,11008687,2,60000001,3,1100205,Porto Velho,0,0,17,NaN,NaN,0,MUNICIPAL,2º Ano Do Ensino Fundamental
3,2023,11,RO,11008682,2,60000001,3,1100205,Porto Velho,1,1,17,1.045465,759.206313,1,MUNICIPAL,2º Ano Do Ensino Fundamental
4,2023,11,RO,11008729,2,60000001,3,1100205,Porto Velho,0,0,19,NaN,NaN,0,MUNICIPAL,2º Ano Do Ensino Fundamental
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2023,11,RO,11008798,2,60000002,2,1100205,Porto Velho,1,1,15,1.057721,744.615008,1,ESTADUAL,2º Ano Do Ensino Fundamental
96,2023,11,RO,11008791,2,60000002,2,1100205,Porto Velho,1,1,15,1.057721,737.671062,0,ESTADUAL,2º Ano Do Ensino Fundamental
97,2023,11,RO,11008813,2,60000002,2,1100205,Porto Velho,1,1,16,0.937304,768.697866,1,ESTADUAL,2º Ano Do Ensino Fundamental
98,2023,11,RO,11008789,2,60000002,2,1100205,Porto Velho,1,1,15,1.057721,782.794855,1,ESTADUAL,2º Ano Do Ensino Fundamental


In [26]:
# fato de alfabetizaçao por escola, municipio e estado, com informações de proficiência e presença dos alunos.
df_fato = (
    df_alunos
    .groupby(
        [
            "NU_ANO_AVALIACAO",
            "CO_UF",
            "CO_MUNICIPIO",
            "ID_ESCOLA",
            'DESC_DEPENDENCIA',
            'DESC_SERIE'
        ],
        as_index=False
    )
    .agg(
        QT_ALUNOS=('ID_ALUNO', 'count'),
        QT_PRESENTES_LP=('IN_PRESENCA_LP', 'sum'),
        QT_PREENCHIMENTO_LP=('IN_PREENCHIMENTO_LP', 'sum'),
        QT_ALFABETIZADOS=('IN_ALFABETIZADO', 'sum'),
        VL_MEAN_PROFICIENCIA_LP=('VL_PROFICIENCIA_LP', 'mean'),
        VL_MIN_PROFICIENCIA_LP=('VL_PROFICIENCIA_LP', 'min'),
        VL_MAX_PROFICIENCIA_LP=('VL_PROFICIENCIA_LP', 'max'),
        VL_MED_PROFICIENCIA_LP=('VL_PROFICIENCIA_LP', 'median'),
        VL_STD_PROFICIENCIA_LP=('VL_PROFICIENCIA_LP', 'std'),
        VL_VAR_PROFICIENCIA_LP=('VL_PROFICIENCIA_LP', 'var')
       
    )   
)   

### Outras métricas
Abstenção
Quartiles
Amplitude
Rankings

In [27]:
df_fato["ABSTENCAO_LP"] = (
    100 * (1 - df_fato["QT_PRESENTES_LP"] / df_fato["QT_ALUNOS"])
).round(2)

df_fato["P25_PROFICIENCIA_LP"] = (
    df_alunos["VL_PROFICIENCIA_LP"].quantile(0.25)
).round(2)

df_fato["P75_PROFICIENCIA_LP"] = (
    df_alunos["VL_PROFICIENCIA_LP"].quantile(0.75)
).round(2)

df_fato["P90_PROFICIENCIA_LP"] = (
    df_alunos["VL_PROFICIENCIA_LP"].quantile(0.9)
).round(2)

df_fato["AMP_PROFICIENCIA_LP"] = (
    df_fato["VL_MAX_PROFICIENCIA_LP"] - df_fato["VL_MIN_PROFICIENCIA_LP"]
).round(2)

df_fato["RANKING_MUNICIPIO"] = (
    df_fato.groupby(
        [
            "NU_ANO_AVALIACAO",
            "CO_MUNICIPIO"
        ]
    )["VL_MEAN_PROFICIENCIA_LP"]
    .rank(
        ascending=False
    )
)

df_fato["RANKING_ESTADO"] = (
    df_fato.groupby(
        [
            "NU_ANO_AVALIACAO",
            "CO_UF"
        ]
    )["VL_MEAN_PROFICIENCIA_LP"]
    .rank(
        ascending=False
    )
)

df_fato["RANKING_BRASIL"] = (
    df_fato.groupby(
        [
            "NU_ANO_AVALIACAO"
            
        ]
    )["VL_MEAN_PROFICIENCIA_LP"]
    .rank(
        ascending=False
    )
)

In [30]:
df_fato.head(1000)

,NU_ANO_AVALIACAO,CO_UF,CO_MUNICIPIO,ID_ESCOLA,DESC_DEPENDENCIA,DESC_SERIE,QT_ALUNOS,QT_PRESENTES_LP,QT_PREENCHIMENTO_LP,QT_ALFABETIZADOS,VL_MEAN_PROFICIENCIA_LP,VL_MIN_PROFICIENCIA_LP,VL_MAX_PROFICIENCIA_LP,VL_MED_PROFICIENCIA_LP,VL_STD_PROFICIENCIA_LP,VL_VAR_PROFICIENCIA_LP,ABSTENCAO_LP,P25_PROFICIENCIA_LP,P75_PROFICIENCIA_LP,P90_PROFICIENCIA_LP,AMP_PROFICIENCIA_LP,RANKING_MUNICIPIO,RANKING_ESTADO,RANKING_BRASIL
0,2023,11,1100015,60000156,MUNICIPAL,2º Ano Do Ensino Fundamental,16,4,4,0,717.176837,679.698133,740.698060,724.155578,26.334837,693.523619,75.0,722.69,782.01,805.7,61.00,5.0,380.0,31926.0
1,2023,11,1100015,60000159,MUNICIPAL,2º Ano Do Ensino Fundamental,26,24,24,16,761.998480,713.273897,804.854173,764.159096,29.675451,880.632372,7.69,722.69,782.01,805.7,91.58,2.0,163.0,9314.0
2,2023,11,1100015,60000161,MUNICIPAL,2º Ano Do Ensino Fundamental,46,45,45,27,742.328474,640.637477,821.453266,747.657988,39.163747,1533.799063,2.17,722.69,782.01,805.7,180.82,4.0,290.0,20372.0
3,2023,11,1100015,60000337,MUNICIPAL,2º Ano Do Ensino Fundamental,91,84,84,65,772.516537,677.351389,835.014756,780.660096,43.073484,1855.325029,7.69,722.69,782.01,805.7,157.66,1.0,107.0,5449.0
4,2023,11,1100015,60000382,MUNICIPAL,2º Ano Do Ensino Fundamental,75,70,70,45,757.900317,644.446169,846.516484,750.784504,48.060770,2309.837620,6.67,722.69,782.01,805.7,202.07,3.0,188.0,11259.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,2023,13,1302603,60001137,MUNICIPAL,2º Ano Do Ensino Fundamental,197,160,160,80,735.082995,627.553651,810.417792,742.074316,40.424496,1634.139893,18.78,722.69,782.01,805.7,182.86,210.0,484.0,24483.0
996,2023,13,1302603,60001138,MUNICIPAL,2º Ano Do Ensino Fundamental,109,64,64,32,731.096620,630.759789,803.441636,744.478510,39.335437,1547.276594,41.28,722.69,782.01,805.7,172.68,234.0,550.0,26446.0
997,2023,13,1302603,60001139,MUNICIPAL,2º Ano Do Ensino Fundamental,211,166,166,71,722.481559,629.127597,810.307238,732.988565,43.185478,1864.985508,21.33,722.69,782.01,805.7,181.18,301.0,686.0,30151.0
998,2023,13,1302603,60001140,ESTADUAL,2º Ano Do Ensino Fundamental,75,59,59,26,727.367024,631.700837,812.740107,732.992512,42.197576,1780.635401,21.33,722.69,782.01,805.7,181.04,265.0,619.0,28179.0


In [ ]:

# Merge com as dimensões de municípios e estados para enriquecer o DataFrame fato com informações adicionais.
df_fato_full = (
    df_alunos
    .merge(
        df_municipios[
            [
                'NU_ANO_AVALIACAO', 'CO_MUNICIPIO', 
                'PC_ALUNO_NIVEL_0_LP', 'PC_ALUNO_NIVEL_1_LP',
                'PC_ALUNO_NIVEL_2_LP', 'PC_ALUNO_NIVEL_3_LP', 'PC_ALUNO_NIVEL_4_LP',
                'PC_ALUNO_NIVEL_5_LP', 'PC_ALUNO_NIVEL_6_LP', 'PC_ALUNO_NIVEL_7_LP',
                'PC_ALUNO_NIVEL_8_LP'
            ]
        ].rename(columns={
                'PC_ALUNO_NIVEL_0_LP': 'MUN_PC_ALUNO_NIVEL_0_LP',
                'PC_ALUNO_NIVEL_1_LP': 'MUN_PC_ALUNO_NIVEL_1_LP',
                'PC_ALUNO_NIVEL_2_LP': 'MUN_PC_ALUNO_NIVEL_2_LP',
                'PC_ALUNO_NIVEL_3_LP': 'MUN_PC_ALUNO_NIVEL_3_LP',
                'PC_ALUNO_NIVEL_4_LP': 'MUN_PC_ALUNO_NIVEL_4_LP',
                'PC_ALUNO_NIVEL_5_LP': 'MUN_PC_ALUNO_NIVEL_5_LP',
                'PC_ALUNO_NIVEL_6_LP': 'MUN_PC_ALUNO_NIVEL_6_LP',
                'PC_ALUNO_NIVEL_7_LP': 'MUN_PC_ALUNO_NIVEL_7_LP',
                'PC_ALUNO_NIVEL_8_LP': 'MUN_PC_ALUNO_NIVEL_8_LP'
            }
        ),
        left_on=[
            "NU_ANO_AVALIACAO",
            "TP_DEPENDENCIA",
            "CO_MUNICIPIO"	
        ],
        right_on=[
            "NU_ANO_AVALIACAO",
            "ID_TIPO_REDE",
            "CO_MUNICIPIO"	
        ],
        how="left"
    )
    .merge(
        df_estados[
            [
                'NU_ANO_AVALIACAO', 'CO_UF', 'ID_TIPO_REDE',
                'PC_ALUNO_ALFABETIZADO', 'VL_MEDIA_LP', 
                'PC_ALUNO_NIVEL_0_LP', 'PC_ALUNO_NIVEL_1_LP', 'PC_ALUNO_NIVEL_2_LP',
                'PC_ALUNO_NIVEL_3_LP', 'PC_ALUNO_NIVEL_4_LP', 'PC_ALUNO_NIVEL_5_LP',
                'PC_ALUNO_NIVEL_6_LP', 'PC_ALUNO_NIVEL_7_LP', 'PC_ALUNO_NIVEL_8_LP'
            ]
        ].rename(columns={
                'PC_ALUNO_ALFABETIZADO': 'EST_PC_ALUNO_ALFABETIZADO',
                'VL_MEDIA_LP': 'EST_VL_MEDIA_LP',
                'PC_ALUNO_NIVEL_0_LP': 'EST_PC_ALUNO_NIVEL_0_LP',
                'PC_ALUNO_NIVEL_1_LP': 'EST_PC_ALUNO_NIVEL_1_LP',
                'PC_ALUNO_NIVEL_2_LP': 'EST_PC_ALUNO_NIVEL_2_LP',
                'PC_ALUNO_NIVEL_3_LP': 'EST_PC_ALUNO_NIVEL_3_LP',
                'PC_ALUNO_NIVEL_4_LP': 'EST_PC_ALUNO_NIVEL_4_LP',
                'PC_ALUNO_NIVEL_5_LP': 'EST_PC_ALUNO_NIVEL_5_LP',
                'PC_ALUNO_NIVEL_6_LP': 'EST_PC_ALUNO_NIVEL_6_LP',
                'PC_ALUNO_NIVEL_7_LP': 'EST_PC_ALUNO_NIVEL_7_LP',
                'PC_ALUNO_NIVEL_8_LP': 'EST_PC_ALUNO_NIVEL_8_LP'
            }
        ),
        left_on=[
            "NU_ANO_AVALIACAO",
            "TP_DEPENDENCIA",
            "CO_UF"	
        ],
        right_on=[
            "NU_ANO_AVALIACAO",
            "ID_TIPO_REDE",
            "CO_UF"	
        ],
        how="left"   
    )
    .merge(
        df_itens[
            [
                'NU_ANO_AVALIACAO', 'CO_UF', 
                'CO_ITEM', 'NU_DESCRITOR_HABILIDADE',
                'DS_GABARITO', 'TP_RESPOSTA_ITEM', 'TP_MODELO_TRI', 'NU_PARAM_A',
                'NU_PARAM_B', 'NU_PARAM_C', 'NU_PARAM_B1', 'NU_PARAM_B2', 'NU_PARAM_B3',
                'NU_PARAM_B4', 'IN_ITEM_COMUM', 'DESC_DISCIPLINA'
            ]
        ].rename(columns={
            'CO_ITEM': 'ITE_CO_ITEM',
            'NU_DESCRITOR_HABILIDADE': 'ITE_NU_DESCRITOR_HABILIDADE',
            'DS_GABARITO': 'ITE_DS_GABARITO',
            'TP_RESPOSTA_ITEM': 'ITE_TP_RESPOSTA_ITEM',
            'TP_MODELO_TRI': 'ITE_TP_MODELO_TRI',
            'NU_PARAM_A': 'ITE_NU_PARAM_A',
            'NU_PARAM_B': 'ITE_NU_PARAM_B',
            'NU_PARAM_C': 'ITE_NU_PARAM_C',
            'NU_PARAM_B1': 'ITE_NU_PARAM_B1',
            'NU_PARAM_B2': 'ITE_NU_PARAM_B2',
            'NU_PARAM_B3': 'ITE_NU_PARAM_B3',
            'NU_PARAM_B4': 'ITE_NU_PARAM_B4',
            'IN_ITEM_COMUM': 'ITE_IN_ITEM_COMUM'
        }),
        left_on=[
            "NU_ANO_AVALIACAO",
            "CO_UF"	
        ],
        right_on=[
            "NU_ANO_AVALIACAO",
            "CO_UF"	
        ],
        how="left"  
    )    
)   


KeyError: 'ID_TIPO_REDE'

In [ ]:
df_fato_full.head()

,NU_ANO_AVALIACAO,CO_UF_x,SG_UF_x,ID_ALUNO,TP_SERIE_x,ID_ESCOLA,TP_DEPENDENCIA,CO_MUNICIPIO,NO_MUNICIPIO_x,IN_PRESENCA_LP,IN_PREENCHIMENTO_LP,CO_CADERNO_LP,VL_PESO_ALUNO_LP,VL_PROFICIENCIA_LP,IN_ALFABETIZADO,DESC_DEPENDENCIA,DESC_SERIE_x,ANO_x,CO_BLOCO_1,TX_RESPOSTA_BLOCO_1,TX_GABARITO_BLOCO_1,CO_BLOCO_2,TX_RESPOSTA_BLOCO_2,TX_GABARITO_BLOCO_2,CO_BLOCO_3,TX_RESPOSTA_BLOCO_3,TX_GABARITO_BLOCO_3,CO_BLOCO_4,TX_RESPOSTA_BLOCO_4,TX_GABARITO_BLOCO_4,CO_UF_y,SG_UF_y,NO_MUNICIPIO_y,TP_SERIE_y,ID_TIPO_REDE,PC_ALUNO_ALFABETIZADO,VL_MEDIA_LP,DESC_SERIE_y,ANO_y,PC_ALUNO_NIVEL_0_LP,PC_ALUNO_NIVEL_1_LP,PC_ALUNO_NIVEL_2_LP,PC_ALUNO_NIVEL_3_LP,PC_ALUNO_NIVEL_4_LP,PC_ALUNO_NIVEL_5_LP,PC_ALUNO_NIVEL_6_LP,PC_ALUNO_NIVEL_7_LP,PC_ALUNO_NIVEL_8_LP
0,2023,11,RO,11008701,2,60000001,3,1100205,Porto Velho,0,0,17,NaN,NaN,0,MUNICIPAL,2º Ano Do Ensino Fundamental,2023,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023,11,RO,11008695,2,60000001,3,1100205,Porto Velho,1,1,17,1.045465,714.314857,0,MUNICIPAL,2º Ano Do Ensino Fundamental,2023,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023,11,RO,11008687,2,60000001,3,1100205,Porto Velho,0,0,17,NaN,NaN,0,MUNICIPAL,2º Ano Do Ensino Fundamental,2023,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2023,11,RO,11008682,2,60000001,3,1100205,Porto Velho,1,1,17,1.045465,759.206313,1,MUNICIPAL,2º Ano Do Ensino Fundamental,2023,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2023,11,RO,11008729,2,60000001,3,1100205,Porto Velho,0,0,19,NaN,NaN,0,MUNICIPAL,2º Ano Do Ensino Fundamental,2023,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
    .merge(
        df_estados,
        left_on=[
            "NU_ANO_AVALIACAO",
            "TP_DEPENDENCIA",
            "CO_UF"	
        ],
        right_on=[
            "NU_ANO_AVALIACAO",
            "ID_TIPO_REDE",
            "CO_UF"	
        ],
        how="left"   
    )
    .merge(
        df_itens,
        left_on=[
            "NU_ANO_AVALIACAO",
            "CO_UF"	
        ],
        right_on=[
            "NU_ANO_AVALIACAO",
            "CO_UF"	
        ],
        how="left"  
    )    
)   


In [ ]:
# Trazendo as colunas que serão utilizadas para a análise, filtrando dos DataFrames carregados da camada Gold.
colunas_utilizadas = [
    'NU_ANO_AVALIACAO', 'ID_ALUNO', 
       'ID_ESCOLA', 'CO_MUNICIPIO',
       'IN_PRESENCA_LP', 'IN_PREENCHIMENTO_LP', 'CO_CADERNO_LP',
       'VL_PESO_ALUNO_LP', 'VL_PROFICIENCIA_LP', 'IN_ALFABETIZADO',
       'DESC_DEPENDENCIA', 'DESC_SERIE'
]

df_alunos = df_alunos[colunas_utilizadas]